# 02 — Descriptive Analysis

**Project:** Private and Public Good Provision of Mental Health Services in France  

---

## Overview

This notebook produces the descriptive statistics and exploratory visualisations used in the paper. It covers three main analyses:

1. **Correlation matrix** — pairwise correlations between all numeric variables in the therapist dataset, to identify which features co-vary and detect potential multicollinearity before regression.
2. **Binned scatter plots (binsreg)** — non-parametric visualisation of the relationship between therapist pricing and distance from Paris, for both in-person and online consultations. Binned scatters are preferred over raw scatter plots when the number of observations is large, as they reveal the conditional mean function without noise.
3. **Service specialty breakdown** — counts and percentages of therapists offering each specialty, both at the national level and disaggregated by department, including identification of the most and least prevalent services per department.

**Input:** `data/processed/full_therapists.geojson`  
**Outputs:** `data/processed/service_counts.csv`, `data/processed/min_max_therapist.xlsx`

---

## Setup

In [ ]:
# Install binsreg if not already available
# !pip install binsreg

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
from binsreg import binsreg

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

---

## 1. Load Data

We load the processed GeoJSON file containing one record per therapist, enriched with geographic and administrative identifiers (department code `INSEE_DEP`, latitude/longitude, etc.).

In [ ]:
therapists_full = gpd.read_file("data/processed/full_therapists.geojson")

# Cast Rating to integer (it is stored as float in the raw data)
therapists_full["Rating"] = therapists_full["Rating"].astype(int)

print(f"Dataset: {len(therapists_full):,} rows, {therapists_full['id'].nunique():,} unique therapists")
print("\nColumns:", list(therapists_full.columns))

---

## 2. Correlation Matrix

We compute pairwise Pearson correlations across all numeric columns to understand how features relate to one another. The geometry column is dropped prior to computation since it is non-numeric.

**Key variables of interest:**
- `service_price` / `service_online_price` — consultation fees
- `reviews_rating` / `reviews_count` — patient feedback signals
- `is_online`, `is_client`, `contactable` — platform engagement indicators
- `position_numeric`, `visibility_numeric` — ranking and visibility scores assigned by the platform

In [ ]:
# Drop geometry column and compute correlation matrix on numeric columns only
df_numeric = therapists_full.drop(columns=["geometry"], errors="ignore")
corr_matrix = df_numeric.corr(numeric_only=True)

# Plot the lower-triangle heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    vmin=-1, vmax=1,
    ax=ax
)
ax.set_title("Correlation Matrix — Therapist Platform Features", fontsize=14, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig("figures/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("""
Key insight: Online presence (is_online) shows strong positive correlation
with visibility_numeric (0.88) and is_client (0.74), suggesting that therapists
who offer online consultations are significantly more prominent on the platform.
""")

---

## 3. Binned Scatter Plots: Pricing vs. Distance from Paris

### Rationale

We hypothesise that geography shapes private therapy pricing: therapists in and around Paris face stronger competition, exerting downward pressure on prices, while therapists in remote areas act more as price-makers. To test this, we visualise the relationship between session price and distance from Paris using **binned scatter plots** via the `binsreg` library.

Binned scatters partition the x-axis into equal-frequency bins and plot the conditional mean of y within each bin, with a polynomial trend line overlaid. This approach is more informative than a raw scatter when observations number in the thousands.

### Data Preparation

We compute each therapist's geodesic distance from Paris (48.8566°N, 2.3522°E) and filter out implausible outliers (distance > 1,000 km, likely overseas territories or data errors).

In [ ]:
# Reload and clean the dataset for this analysis
therapists_full = gpd.read_file("data/processed/full_therapists.geojson")
therapists_full = therapists_full.dropna(subset=["lat", "long", "service_price"])

PARIS = (48.8566, 2.3522)

therapists_full["distance_to_paris"] = therapists_full.apply(
    lambda row: geodesic((row["lat"], row["long"]), PARIS).km, axis=1
)

# Remove extreme outliers (overseas territories, geocoding errors)
therapists_filtered = therapists_full[therapists_full["distance_to_paris"] < 1000].copy()

print(f"Observations after filtering: {len(therapists_filtered):,}")
print(f"Distance range: {therapists_filtered['distance_to_paris'].min():.1f} — "
      f"{therapists_filtered['distance_to_paris'].max():.1f} km")

### 3.1 — In-Person Session Price vs. Distance

In [ ]:
# Binned scatter: in-person price ~ distance from Paris
# 50 equal-frequency bins, linear polynomial fit within each bin
est_inperson = binsreg(
    "service_price",
    "distance_to_paris",
    data=therapists_filtered,
    nbins=50,
    polyreg=1
)

print(est_inperson.bins_plot)

### 3.2 — Online Session Price vs. Distance

Online prices should theoretically be less geographically determined since the therapist's physical location matters less. We run the same binned regression for online prices to compare the gradient.

In [ ]:
# Keep only therapists with an online price listed
therapists_online = therapists_filtered.dropna(subset=["service_online_price"])

est_online = binsreg(
    "service_online_price",
    "distance_to_paris",
    data=therapists_online,
    nbins=50,
    polyreg=1
)

print(est_online.bins_plot)

print("""
Key insight: Both in-person and online prices show a negative gradient with
distance from Paris, but the slope is flatter for online prices, consistent
with online consultations being less geographically constrained.
""")

---

## 4. Service Specialty Breakdown

Each therapist lists one or more specialties (e.g. Depression, Anxiety, Autism). The `Services` column contains a comma-separated string of specialties per therapist. We explode this into long format to count how many therapists offer each service.

### 4.1 — National-Level Service Counts

In [ ]:
therapists_full = gpd.read_file("data/processed/full_therapists.geojson")

# Split comma-separated service strings into lists, then explode to long format
therapists_full["Services"] = therapists_full["Services"].astype(str).str.split(",")
exploded = therapists_full.explode("Services")
exploded["Services"] = exploded["Services"].str.strip()

total_therapists = therapists_full["Name"].nunique()

service_counts = (
    exploded
    .groupby("Services")["id"]
    .nunique()
    .reset_index(name="therapist_count")
)
service_counts["percentage"] = service_counts["therapist_count"] / total_therapists * 100
service_counts.sort_values("therapist_count", ascending=False, inplace=True)

print(f"Total unique therapists: {total_therapists:,}")
print(f"Total unique services listed: {len(service_counts)}\n")
print(service_counts.head(20).to_markdown(index=False))

# Save to disk
import os
os.makedirs("data/processed", exist_ok=True)
service_counts.to_csv("data/processed/service_counts.csv", index=False)

### 4.2 — Most and Least Offered Services by Department

We now disaggregate to the department level (`INSEE_DEP`) to identify which service is the most prevalent and which is the least prevalent in each administrative unit. This output feeds directly into the choropleth maps produced in notebook 03.

In [ ]:
# Count unique therapists per department × service combination
dept_service = (
    exploded
    .groupby(["INSEE_DEP", "Services"])["id"]
    .nunique()
    .reset_index(name="therapist_count")
)

# Total therapists per department (for percentage calculation)
dept_total = (
    therapists_full
    .groupby("INSEE_DEP")["id"]
    .nunique()
    .reset_index(name="total_therapists")
)

dept_service = dept_service.merge(dept_total, on="INSEE_DEP")
dept_service["percentage"] = dept_service["therapist_count"] / dept_service["total_therapists"] * 100

In [ ]:
# Most offered service per department (by raw therapist count)
most_offered = (
    dept_service
    .loc[dept_service.groupby("INSEE_DEP")["therapist_count"].idxmax()]
    .rename(columns={
        "Services": "most_treated_service",
        "therapist_count": "max_therapist_count"
    })[["INSEE_DEP", "most_treated_service", "max_therapist_count"]]
)

# Least offered service per department
least_offered = (
    dept_service
    .loc[dept_service.groupby("INSEE_DEP")["therapist_count"].idxmin()]
    .rename(columns={
        "Services": "least_treated_service",
        "therapist_count": "min_therapist_count"
    })[["INSEE_DEP", "least_treated_service", "min_therapist_count"]]
)

min_max_summary = most_offered.merge(least_offered, on="INSEE_DEP")
print(f"Department-level summary: {len(min_max_summary)} departments")
min_max_summary.head(10)

In [ ]:
# Save to Excel — this file is used as input in notebook 03
min_max_summary.to_excel("data/processed/min_max_therapist.xlsx", index=False)
print("Saved: data/processed/min_max_therapist.xlsx")

### 4.3 — Summary Statistics by Urban Density Label

INSEE classifies each commune into a density category ranging from highly rural to large urban centres. We summarise average ratings and prices by density label to understand whether urban concentration affects both the quantity and cost of private therapy.

In [ ]:
# Descriptive statistics by density label (requires 'density_label' column in the GeoJSON)
if "density_label" in therapists_full.columns:
    summary = (
        therapists_full
        .groupby("density_label")
        .agg(
            avg_rating=("Rating", "mean"),
            std_rating=("Rating", "std"),
            avg_price=("service_price", "mean"),
            std_price=("service_price", "std"),
            n_appointments=("id", "count"),
        )
        .assign(
            pct_appointments=lambda df: df["n_appointments"] / df["n_appointments"].sum() * 100
        )
    )
    print(summary.to_markdown())
else:
    print("'density_label' column not found. Skipping density summary.")